# Day 2 · Module 1 — DNA as a Language (Sequence Modeling Playground)

**Driving question:** *If a model has read the genome of all of life but was never told which sequences are "good," can its sense of "this DNA looks normal" carry real biological meaning?*

**Objective:** get a **likelihood** out of a DNA language model (HyenaDNA-small, CPU-ok) and learn to read it biologically — because a variant's effect (Module 2) is just *a subtraction of two of these likelihoods*.

> **Student notebook** · kernel **AImed**. No pass/retry gate here — like Day-1 M1, the deliverable is a **defended DECISION**; a CHECKPOINT cell plots the answer if you get stuck.

### Pre-work recap
- **Autoregressive language model:** predicts the next token from everything before it; you saw this for English words in the video.
- **Perplexity:** how *surprised* a model is, on average, per token. Low perplexity = the model finds the text predictable.
- *Arrive-with:* What would "perplexity" even mean for a string of **A / C / G / T**? Hold that question — you'll compute it in this notebook.
- **The one idea the whole day rests on:** *variant effect = a subtraction of two likelihoods* — `delta = logprob(alternate) − logprob(reference)`. Today we build the **logprob** half; Module 2 does the subtraction.

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import dna_lm_playground` and `from common import models`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Guided hands-on

### Background — what is a "likelihood of a sequence," and why would DNA have one?

**The model.** HyenaDNA is a *language model for DNA*. It was trained on the human reference genome the same way a text model is trained on the web: hide the next letter, make the model guess it, repeat billions of times. The only "words" it ever sees are the four DNA bases **A, C, G, T** (plus a few special tokens). After training it has absorbed the statistical *grammar* of real genomes — which letters tend to follow which, what real coding regions look like, what repeats look like — **without anyone ever labeling a sequence as good or bad.**

**Autoregressive = one base at a time.** To read a sequence the model walks left to right. At each position it outputs a probability for what the **next** base will be, given everything to the left. So for `ATG...` it produces `P(next = A | A)`, then `P(next = G | A T)`, and so on.

**The likelihood.** Multiply those next-base probabilities together for the whole sequence — equivalently, **add their logs** — and you get one number: the **log-likelihood** of the sequence.

> `sequence_logprob(seq) = sum over positions of  log P(this base | all bases before it)`

A **higher** (closer-to-zero, less-negative) log-likelihood means *"this string looks like real genome to me."* A **lower** (more-negative) value means *"this surprises me."* That is the whole sensor.

**Key terms.**
- **token** — for HyenaDNA, one DNA letter.
- **log-likelihood (logprob)** — the model's score for how natural a sequence looks; higher = more natural. It is *negative* (you're summing logs of probabilities < 1), and *longer sequences have more-negative scores* simply because there are more terms — so only compare sequences of the **same length**.
- **perplexity** — `exp(−logprob / length)`: average surprise per base. Perplexity near 1 = totally predictable; a DNA model that guessed each base by coin-flip among 4 would have perplexity ~4. **Lower = the model finds the DNA more natural.**

**Analogy — a lifelong reader who has never seen a grammar book.** Imagine someone who has read every book ever written but was never taught grammar rules and never told which sentences are "correct." Hand them a new sentence and they can still tell you *how natural it sounds*. "The cat sat on the mat" feels ordinary; "cat the on sat mat the" feels wrong; random letters feel like nonsense — and they can rate them **without any answer key**. HyenaDNA is that reader, for the language of DNA. The log-likelihood is its gut feeling, turned into a number. Today's question is whether that gut feeling tracks *biology*.

In [ ]:
# Guided compute — load the model once, score one real biological sequence, and PRINT the pieces.
# The helpers in dna_lm_playground.py / common.models do the heavy lifting; we just read the numbers.
from common import models
import dna_lm_playground as PG
import numpy as np

scorer = models.load_dna_model('hyenadna-small-32k')   # HyenaDNA-small, ~6.6M params, CPU-ok
seq = PG.load_human_example(window=300)               # a real human chr17 window in the BRCA1 locus (hg19)
# in-distribution for a human-genome model -> low perplexity; 300 bp keeps CPU scoring quick

ll = scorer.sequence_logprob(seq)                      # total log-likelihood (sum of per-base log-probs)
per_base = scorer.per_base_logprob(seq)                # the individual log P(base | history) terms
perplexity = float(np.exp(-ll / len(seq)))             # average surprise per base

print('sequence length      :', len(seq))
print('total log-likelihood :', round(ll, 2), '  (higher / closer to 0 = more natural; it is negative)')
print('mean per-base logprob :', round(float(per_base.mean()), 3))
print('perplexity            :', round(perplexity, 3), '  (lower = model finds it more predictable)')
print('sanity: sum(per_base) ~= total logprob ->', round(float(per_base.sum()), 2))

### Read what just printed
- The **total log-likelihood** is one number summarizing the whole sequence. It is negative — that is normal (logs of probabilities are negative).
- **`sum(per_base) ~= total logprob`**: the sequence score really is just the sum of the per-position surprises. This is the structure you will exploit in the demo and in *Your Turn*.
- **Perplexity** turns that into an intuitive "average surprise per base." A model guessing each base by coin-flip among 4 would sit at perplexity ~4; because this is **real human DNA and HyenaDNA was trained on the human genome**, it lands *well below 4* — the model genuinely finds real genome predictable. (Score DNA from the *wrong* organism — a virus, say — and the perplexity shoots far *above* 4: "natural" is relative to what the model was trained on.)

In [ ]:
# VIVID DEMO — does the likelihood actually track biology? Make the same-length comparison undeniable.
# We take the SAME real human-genome window from above and compare its likelihood to (a) a frameshifted
# copy (insert one base -> every codon after it slips) and (b) a length-matched RANDOM-base copy.
# SAME LENGTH each time, so the only thing changing is how 'natural' the string looks to the model.
import numpy as np
from common.genomics import insert_frameshift

rng = np.random.default_rng(0)
natural = seq                                          # the real human chr17 window scored above
L = len(natural)
shifted = insert_frameshift(natural)[:L]               # insert one base -> frame slips; trim back to L
random_dna = ''.join(rng.choice(list('ACGT'), size=L)) # same length, no genome grammar at all

ll_natural = scorer.sequence_logprob(natural)
ll_shifted = scorer.sequence_logprob(shifted)
ll_random  = scorer.sequence_logprob(random_dna)

print(f'length (all three) : {L}')
print(f'LL(real human window) : {ll_natural:8.2f}')
print(f'LL(frameshifted copy) : {ll_shifted:8.2f}   drop vs natural: {ll_natural - ll_shifted:+.2f}')
print(f'LL(random A/C/G/T)    : {ll_random:8.2f}   drop vs natural: {ll_natural - ll_random:+.2f}')
print()
print('Takeaway: with NO labels, the real human window scores HIGHEST and the random copy LOWEST -- a')
print('big gap. The frameshift drop is SMALL: this 6.6M-param model barely notices a slipped frame --')
print('a first hint at why it scores near chance on real BRCA1 variants in Module 2 (scale fixes that).')

In [ ]:
# YOUR TURN — compute perplexity yourself, the long way, to prove you understand the sum-of-logprobs.
# No black box: you will rebuild the model's score from its per-base parts. Fill each  = ...
import numpy as np

# Use the real human 'natural' window from the demo cell.
pb = scorer.per_base_logprob(natural)         # array of per-position log P(base | history)

# (a) Rebuild the total log-likelihood by SUMMING the per-base log-probs.
my_total_ll = ...                              # TODO: sum the per-base array  (hint: pb.sum())

# (b) Perplexity = exp( - mean per-base log-prob ).  mean = total_ll / length.
n_scored = len(pb)                             # number of per-base terms
my_perplexity = ...                            # TODO: float(np.exp(- my_total_ll / n_scored))

# (c) Do the same for the RANDOM sequence and compare.
pb_rand = scorer.per_base_logprob(random_dna)
rand_perplexity = ...                          # TODO: float(np.exp(- pb_rand.sum() / len(pb_rand)))

print('my rebuilt total LL  :', round(float(my_total_ll), 2))
print('perplexity (natural) :', round(float(my_perplexity), 3))
print('perplexity (random)  :', round(float(rand_perplexity), 3))
print('Lower perplexity = the model finds it MORE natural. Which sequence wins, and why?')

In [ ]:
# Optional visual — WHERE is the model surprised? Plot per-base log-prob along the sequence.
# Spikes downward = positions the model did NOT expect (e.g. right after the frame breaks).
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
pb_nat = scorer.per_base_logprob(natural)
pb_shift = scorer.per_base_logprob(shifted)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(pb_nat, color='C0', lw=1, label='real human window')
ax.plot(pb_shift, color='C3', lw=1, alpha=0.8, label='frameshifted')
ax.set(xlabel='position along sequence (base)', ylabel='log P(base | history)',
       title='Per-base surprise: lower = the model did not expect this base')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()
print('Read it: the frameshifted curve sits LOWER on average -> more total surprise -> lower likelihood.')

### Your task — decide whether you'd trust this likelihood as a biology signal
You now have three numbers (real human window / frameshifted / random) and a perplexity you computed by hand. **Module 2 will turn this exact `sequence_logprob` into a pathogenicity score for real BRCA1 variants.** Before you build that, take a position in the `DECISION` cell: is the likelihood gap you saw *big enough and in the right direction* to bet a variant-effect method on — and what is the one assumption that makes it valid? There is no single right answer; what matters is the reasoning, which you defend in the discussion.

In [ ]:
# YOUR TURN — defend a position. No auto-grader; argue this in the discussion, so be specific
# (cite a likelihood gap, a perplexity, or a comparison from the demo).
DECISION = {
    'likelihood_tracks_biology': ...,   # True / False: did higher likelihood line up with 'more natural' DNA?
    'evidence': ...,                    # one sentence citing your actual numbers (the LL gaps or perplexities)
    'fair_comparison_rule': ...,        # one sentence: the rule that made the comparison valid (hint: length)
    'biggest_risk': ...,                # one sentence: why 'looks natural' might NOT equal 'not pathogenic'
}
print(DECISION)

In [ ]:
# CHECKPOINT (non-blocking) — stuck? This VISUALIZES the answer so you can read it off.
# Uses the backing script's frameshift_demo on a real human chr17 window, plus the demo numbers above.
%matplotlib inline
import matplotlib.pyplot as plt, numpy as np
import dna_lm_playground as PG
from common import models

_fs = PG.frameshift_demo(models.load_dna_model('hyenadna-small-32k'))
print('frameshift_demo:', {k: (round(v, 2) if isinstance(v, float) else v) for k, v in _fs.items()})

# Bar chart of the three same-length likelihoods from the demo cell (re-scored here so the
# checkpoint runs standalone if you skipped ahead).
labels = ['real human\nwindow', 'frameshifted', 'random\nA/C/G/T']
vals = [scorer.sequence_logprob(natural), scorer.sequence_logprob(shifted), scorer.sequence_logprob(random_dna)]
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, vals, color=['C0', 'C1', 'C3'])
ax.set(ylabel='total log-likelihood (higher = more natural)',
       title='Same length, different naturalness -> different likelihood')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:.0f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()
print('Read it off: tallest (least negative) bar = the sequence the model finds most natural.')

**How to read the checkpoint plot.**

Three bars, three sequences of the **same length** — so length is not what's driving the difference; *naturalness* is. The bar height is the total log-likelihood (more negative = lower = more surprising to the model).

**Analogy — the lifelong reader rating three passages.** Same word-count each time: a real sentence (`real human window`), the same sentence with one letter shoved in so every word after it slips (`frameshifted`), and a random jumble of letters (`random`). The reader rates the real one highest and the jumble far lowest, *with no answer key* — that ordering is the whole sensor. The frameshift sits only *slightly* below the real window: a small model barely notices a slipped frame (the honest crack you'll watch widen in Module 2). Your `DECISION['likelihood_tracks_biology']` should be **True** if the real-DNA bar is clearly tallest and the random one clearly shortest — read it straight off the chart. The `fair_comparison_rule` you were asked for is exactly why all three bars share a length.

### Discussion (peer instruction — vote, argue, re-vote)
- **Why should the *likelihood* of a DNA sequence correlate with anything biological at all?** (Selection -> conservation -> the model learns that real-genome patterns are predictable; disruptive sequences are rare in training, so they look "surprising.")
- The model saw **no human disease labels** — only raw genomes. Why might training across all of life *help* human variant prediction rather than hurt it?
- HyenaDNA-small reads ~32,000 bases of context; the frontier models (Evo 2) read ~1,000,000. **Name a kind of variant whose effect you could *only* catch with the longer context** (hint: distant regulatory elements, large structural changes). What does that say about the *limits* of the small model you just ran?

### Stretch (fast finishers only)
1. **Generation probe.** Try `scorer.generate('ATGGCG', max_new_tokens=80)`. The HyenaDNA `-hf` checkpoint may not support generation (the helper returns a friendly message instead of crashing) — if so, explain *why scoring still works even when generation does not* (you only need the model's logits, not a generation head).
2. **Context matters.** Use `PG.context_demo(scorer)` to score a short (128 bp) vs a long (1024 bp) window and discuss how context length changes the likelihood.
3. **Exon/intron structure.** Plot `scorer.per_base_logprob` across a longer real sequence and look for regions of consistently lower surprise — biological structure sometimes shows up as bands in the curve.